# Import

In [1]:
import models.juanchitocnn
import models.visiontransformer
import models.efficientnetb0
import data_loader
import torch
from FGSM_attack import FGSMAttacker
print('import done')

/storage/ice1/8/1/jguardia7/dl-final-project/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-04-27 01:04:59.032832: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-27 01:05:06.811123: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745730307.572745  802500 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745730307.842364  802500 cud

import done


# Settings and Hyperparameters

In [2]:
select_model = 'JuanchitoCNN' # ['EfficientNet', 'VisionTransformer', 'JuanchitoCNN']
train_original = False # Re-train model?
train_augmentation = False 
train_adversarial = False
train_augmentation_adversarial = True

# Model Hyperparameters
epochs = 5
learning_rate = 0.001
batch_size = 32
weight_decay = 0.01
momentum = 0.9
optimizer = 'SGD' # ['SGD', 'AdamW']

# Other parameters
train_test_split_ratio = 0.8

# Only need to run cells below

## Selecting Model

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if select_model == 'EfficientNet':
    project_model = models.efficientnetb0.ProjectEfficientNet(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
    project_model_aug = models.efficientnetb0.ProjectEfficientNet(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
    project_model_adv = models.efficientnetb0.ProjectEfficientNet(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
    project_model_aug_adv = models.efficientnetb0.ProjectEfficientNet(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
    print(f'Using model {select_model}')
elif select_model == 'VisionTransformer':
    project_model = models.visiontransformer.ProjectVisionTransformer(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
    project_model_aug = models.visiontransformer.ProjectVisionTransformer(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
    project_model_adv = models.visiontransformer.ProjectVisionTransformer(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
    project_model_aug_adv = models.visiontransformer.ProjectVisionTransformer(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
    print(f'Using model {select_model}')
elif select_model == 'JuanchitoCNN':
    project_model = models.juanchitocnn.ProjectJuanchitoCNN(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
    project_model_aug = models.juanchitocnn.ProjectJuanchitoCNN(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
    project_model_adv = models.juanchitocnn.ProjectJuanchitoCNN(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
    project_model_aug_adv = models.juanchitocnn.ProjectJuanchitoCNN(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
    print(f'Using model {select_model}')
else:
    print('No valid model selected')

Using model JuanchitoCNN


## Train or Load Model on Regular Data

In [4]:
csv_path = '/home/hice1/jguardia7/scratch/datasets/alessandrasala79/ai-vs-human-generated-dataset/versions/4/train.csv'
image_folder = '/home/hice1/jguardia7/scratch/datasets/alessandrasala79/ai-vs-human-generated-dataset/versions/4/train_data'

In [5]:
print(f'Batch Size: {batch_size}, Split Ratio: {train_test_split_ratio}')

if train_original == True:
    regular_train_loader, regular_test_loader = data_loader.data_to_train_test_dataloaders(csv_path=csv_path, 
                                                                                       image_folder=image_folder, 
                                                                                       image_size=(224, 224), 
                                                                                       split_ratio=train_test_split_ratio, 
                                                                                       train_batch_size=batch_size, 
                                                                                       test_batch_size=batch_size)

    project_model.data_load(regular_train_loader, regular_test_loader)
    print('Training on original data set')
    project_model.train()
elif train_original == False:
    project_model.model_load()
    print('Loaded with original data set')
    
if train_augmentation == True:
    regular_train_loader, regular_test_loader = data_loader.data_to_train_test_dataloaders(csv_path=csv_path, 
                                                                                       image_folder=image_folder, 
                                                                                       image_size=(224, 224), 
                                                                                       split_ratio=train_test_split_ratio, 
                                                                                       train_batch_size=batch_size, 
                                                                                       test_batch_size=batch_size)
    aug_train_loader, aug_test_loader = data_loader.data_to_aug_dataloaders(csv_path=csv_path, 
                                                                            image_folder=image_folder, 
                                                                            image_size=(224, 224), 
                                                                            split_ratio=train_test_split_ratio, 
                                                                            train_batch_size=batch_size, 
                                                                            test_batch_size=batch_size)

    project_model_aug.data_load(aug_train_loader, regular_test_loader)
    print('Training on augmented data set')
    project_model_aug.train(description='aug')

elif train_augmentation == False:
    project_model_aug.model_load(description='aug')
    print('Loaded with augmented data set')

Batch Size: 32, Split Ratio: 0.8
Loaded with original data set
Loaded with augmented data set


In [ ]:
if train_adversarial == True:
    regular_train_loader, regular_test_loader = data_loader.data_to_train_test_dataloaders(csv_path=csv_path, 
                                                                            image_folder=image_folder, 
                                                                            image_size=(224, 224), 
                                                                            split_ratio=train_test_split_ratio, 
                                                                            train_batch_size=batch_size, 
                                                                            test_batch_size=batch_size)

    project_model_adv.data_load(regular_train_loader, regular_test_loader)
    print('Adversarial training on original data set')
    project_model_adv.train_adversarial(description='adv')

elif train_adversarial == False:
    project_model_adv.model_load(description='adv')
    print('Loaded with original data set')
    
if train_augmentation_adversarial == True:
    regular_train_loader, regular_test_loader = data_loader.data_to_train_test_dataloaders(csv_path=csv_path, 
                                                                            image_folder=image_folder, 
                                                                            image_size=(224, 224), 
                                                                            split_ratio=train_test_split_ratio, 
                                                                            train_batch_size=batch_size, 
                                                                            test_batch_size=batch_size)
    aug_train_loader, aug_test_loader = data_loader.data_to_aug_dataloaders(csv_path=csv_path, 
                                                                            image_folder=image_folder, 
                                                                            image_size=(224, 224), 
                                                                            split_ratio=train_test_split_ratio, 
                                                                            train_batch_size=batch_size, 
                                                                            test_batch_size=batch_size)

    project_model_aug.data_load(aug_train_loader, regular_test_loader)
    print('Adversarial training on augmented data set')
    project_model_aug.train_adversarial(description='aug_adv')

elif train_augmentation_adversarial == False:
    project_model_aug.model_load(description='aug_adv')
    print('Loaded with augmented data set')

Loaded with original data set


/storage/ice1/8/1/jguardia7/dl-final-project/lib64/python3.9/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/storage/ice1/8/1/jguardia7/dl-final-project/lib64/python3.9/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Adversarial training on augmented data set


Epoch 2/5:  93%|█████████▎| 1867/1999 [08:40<00:37,  3.52batch/s, loss=0.6940]

## Trial 1 - Trained with original train data, Evaluated on original test data

In [ ]:
print('Trial 1 - Trained with original train data, Evaluated on original test data')
regular_train_loader, regular_test_loader = data_loader.data_to_train_test_dataloaders(csv_path=csv_path, 
                                                                                       image_folder=image_folder, 
                                                                                       image_size=(224, 224), 
                                                                                       split_ratio=train_test_split_ratio, 
                                                                                       train_batch_size=batch_size, 
                                                                                       test_batch_size=batch_size)

project_model.data_load(regular_train_loader, regular_test_loader)
regular_correct, total, regular_incorrect_preds = project_model.evaluate()

## Trial 2 - Trained with original train data, Evaluated on perturbed test data

In [ ]:
print('Trial 2 - Trained with original train data, Evaluated on perturbed test data')
for attack_style in ['SaltAndPepper', 'Posterize', 'GaussNoise', 'RandomShadow']:

    attack_train_loader, attack_test_loader = data_loader.data_to_attack_dataloaders(csv_path=csv_path, 
                                                                                     image_folder=image_folder, 
                                                                                     image_size=(224, 224), 
                                                                                     split_ratio=train_test_split_ratio, 
                                                                                     train_batch_size=batch_size, 
                                                                                     test_batch_size=batch_size, 
                                                                                     attack_style=attack_style)
    
    project_model.data_load(None, attack_test_loader)
    print(f'Attack type is {attack_style}')
    attack_correct, total, attack_incorrect_preds = project_model.evaluate()

## Trial 3 - Trained with original train data, Evaluated on FGSM

In [ ]:
print('Trial 3 - Trained with original train data, Evaluated on FGSM')
for fgsm_epsilon in [0.01, 0.05, 0.1]:
    fgsm_train_loader, fgsm_test_loader = data_loader.data_to_train_test_dataloaders(csv_path='data/train.csv', 
                                                                                           image_folder='data/train_data', 
                                                                                           image_size=(224, 224), 
                                                                                           split_ratio=train_test_split_ratio, 
                                                                                           train_batch_size=1, 
                                                                                           test_batch_size=1)
    
    attacker = FGSMAttacker(project_model.model, device)
    acc, adv_examples = attacker.attack(fgsm_test_loader, fgsm_epsilon)

## Trial 4 - Trained with augmented train data, Evaluated on FGSM

In [ ]:
print('Trial 4 - Trained with augmented train data, Evaluated on FGSM')
for fgsm_epsilon in [0.01, 0.05, 0.1]:
    fgsm_train_loader, fgsm_test_loader = data_loader.data_to_train_test_dataloaders(csv_path='data/train.csv', 
                                                                                           image_folder='data/train_data', 
                                                                                           image_size=(224, 224), 
                                                                                           split_ratio=train_test_split_ratio, 
                                                                                           train_batch_size=1, 
                                                                                           test_batch_size=1)
    
    attacker = FGSMAttacker(project_model_aug.model, device)
    acc, adv_examples = attacker.attack(fgsm_test_loader, fgsm_epsilon)